# Vorlesung: Aufgabe

Parsen der Syntax aus der Vorlesung mit F# und Fparsec

```
>>   parser combination
>>.  output des folgenden parser weitergeben
.>>. output von linken und rechtem parser weitergeben (als tuple)
>>?  backtracking. Wen der nachfolgende parser fehlschlägt kan zum beginn des ersten parser zurück gegangen werden (nützlich für choice)
|>>  Funktion mit der die rückgabe eines parsers transformiert werden kann
<|>  oder
choice [] performantere? version von oder

```

In [1]:
#r "nuget: FParsec, 1.1.1"
open FParsec

let test p str =
    match run p str with
    | Success(result, _, _)   -> printfn "Success: %A" result
    | Failure(errorMsg, _, _) -> printfn "Failure: %s" errorMsg

let toString (xs : seq<char>) : string = String.Concat(xs)

test pfloat "1.25"

Installed Packages FParsec, 1.1.1

Success: 1.25


In [2]:


let identifier = 
    let start = letter <|> pchar '_' |>> fun x -> (string) x
    let mid = many (letter <|> digit <|> pchar '_') |>> toString
    start .>>. mid |>> fun (x, y) -> x + y
    
let number =
    many1 digit |>> toString

let spacesl = many (anyOf " \t")

   
let expression, expressionRef = createParserForwardedToRef()
do expressionRef := 
    let arithmetic a : Parser<string, unit> = 
        choice [identifier; number] .>>? spacesl .>>? pchar a .>> spacesl .>>. expression |>> fun (x, y) -> x + (string)a + y
    choice [
        arithmetic '*'
        arithmetic '+'
        identifier
        number
    ]

let assignment = 
    identifier .>>? spacesl .>>? pchar '=' .>> spacesl .>>. expression |>> fun (x, y) -> x + "=" + y

let line = (assignment <|> expression) .>> pchar ';'

let lines = many1 line .>> eof



// test identifier "_test"
test lines "hello_world = 123 + 456 * hello;"

Success: ["hello_world=123+456*hello"]


In [9]:

type LIdentifier = string

type LLiteral = string

type LExpression =
    | Literal of LLiteral
    | Identifier of LIdentifier
    | Operation of LOperation
and LOperation = 
    | Add of (LExpression * LExpression)
    | Multiply of (LExpression * LExpression)

type LAssignment = {
    Identifier: LIdentifier;
    Expression: LExpression;
}


type LStatement =
    | Assignment of LAssignment
    | Expression of LExpression


let identifier = 
    let start = letter <|> pchar '_' |>> fun x -> (string) x
    let mid = many (letter <|> digit <|> pchar '_') |>> toString
    start .>>. mid |>> fun (x, y) -> x + y |> LIdentifier
    
let number = many1 digit |>> toString |>> LLiteral
    

// let spacesl = many (anyOf " \t")
let spacesl = spaces

// ref shenanigans because of recursive use of pexpression
let pexpression, pexpressionRef = createParserForwardedToRef()
do pexpressionRef := 
    let value = choice [
        between (pchar '(') (pchar ')') (spaces >>. pexpression .>> spaces)
        identifier     |>> LExpression.Identifier
        number |>> LExpression.Literal
    ] 

    // first segment must always be a simple value otherwise the parser will recursively loop without consuming any input
    let arithmetic a : Parser<(LExpression * LExpression), unit> = 
        value .>>? spacesl .>>? pchar a .>> spacesl .>>. pexpression
    
    let multiply = arithmetic '*' |>> LOperation.Multiply
    let add      = arithmetic '+' |>> LOperation.Add
    choice [
        multiply |>> LExpression.Operation
        add |>> LExpression.Operation
        value
    ]

let passignment = 
    identifier .>>? spacesl .>>? pchar '=' .>> spacesl .>>. pexpression 
    |>> fun (x, y) -> { Identifier = x; Expression = y }

let pstatement = 
    choice [
        passignment  |>> LStatement.Assignment
        pexpression |>> LStatement.Expression
    ] .>> pchar ';'

let lines = spaces >>. many1 (pstatement .>> spaces) 
let all = lines .>> eof



// test identifier "_test"
// test (spaces >>. line) "_test;"
test lines "calc_42 = __9 + (zZ_1 + 5) * FooBar_42 * bar_7 + q0;"

Success: [Assignment
   { Identifier = "calc_42"
     Expression =
      Operation
        (Add
           (Identifier "__9",
            Operation
              (Multiply
                 (Operation (Add (Identifier "zZ_1", Literal "5")),
                  Operation
                    (Multiply
                       (Identifier "FooBar_42",
                        Operation (Add (Identifier "bar_7", Identifier "q0")))))))) }]


In [12]:
test pstatement"1 + 2 * 3;"

Success: Expression
  (Operation
     (Add (Literal "1", Operation (Multiply (Literal "2", Literal "3")))))


In [13]:
test pstatement"1 * 2 + 3;"

Success: Expression
  (Operation
     (Multiply (Literal "1", Operation (Add (Literal "2", Literal "3")))))


In [8]:
test all """
result99 = acc_2*ACC_2 + spillover7 + bonus_1*3 + inc_0;
calc_42 = __9 * zZ_1 + 5 + FooBar_42 * bar_7 + q0;
_ExprLine + A_1 * bB_2 + cc3 * 7 +      11;
"""

Success: [Assignment
   { Identifier = "result99"
     Expression =
      Operation
        (Multiply
           (Identifier "acc_2",
            Operation
              (Add
                 (Identifier "ACC_2",
                  Operation
                    (Add
                       (Identifier "spillover7",
                        Operation
                          (Multiply
                             (Identifier "bonus_1",
                              Operation (Add (Literal "3", Identifier "inc_0")))))))))) };
 Assignment
   { Identifier = "calc_42"
     Expression =
      Operation
        (Multiply
           (Identifier "__9",
            Operation
              (Add
                 (Identifier "zZ_1",
                  Operation
                    (Add
                       (Literal "5",
                        Operation
                          (Multiply
                             (Identifier "FooBar_42",
                              Operation
                  

# Post mortem

Jede Person beschreibt in der ILIAS-Abgabe individuell(!) die Bearbeitung des jeweiligen Aufgabenblattes
zurückblickend mit ca. 200 bis 400 Wörtern. Gehen Sie dabei aussagekräftig und nachvollziehbar auf folgende Punkte ein: 
 (a) Zusammenfassung: Was wurde gemacht? 
 (b) Implementierungsdetails: Kurze Beschreibung besonders interessanter Aspekte der Umsetzung. 
 (c) Was war der schwierigste Teil bei der Bearbeitung? Wie haben Sie dieses Problem gelöst? 
 (d) Was haben Sie gelernt oder (besser) verstanden? 
 (e) Team: Mit wem haben Sie zusammengearbeitet? 
 (f) Link zum Repo mit der Lösung 



 * a: Ich habe den antlr Parser in C# implementiert. Nach dem hinzufügen der Nuget Pakete für antlr braucht man nichts weiter machen, als eine .g4 Datei zu erstellen. Die Datei wird während des Compilieren automatisch generiert. Ich habe mit einem "Assignment" angefangen. Ich habe die Antlr grammatik erstellt und mithilfe des Rider / IntelliJ plugins Validiert, ob das Beispiel auch tatsächlich so geparsed wird, wie ich es erwartet. Danach habe ich einen Listener geschrieben, welcher mir den Baum formatiert ausgibt. Von da aus habe ich die Grammatik und den Listener Stück für Stück erweitert. Als letztes habe ich einen Visitor implementiert, der einen AST generiert, welcher auch ausgegeben werden kann.
 * b: Ich habe das Beispiel aus der Vorlesung in F# mithilfe der Fparsec Bibliothek implementiert.
    Ich habe die OneOf Bibliothek verwendet, um die "Union" Typen des AST darzustellen.
 * c: Die Visitor Methoden haben einen Generischen Rückgabewert, welcher für alle Methoden gleich ist. In Retrospective glaube ich, dass es sauberer wäre, eigene Methoden mit richtigen Rückgabewerten zu implementieren. Dadurch sollte es möglich sein die menge and Typechecks zu reduzieren und der Code sollte sauberer werden. 
    Ich musste feststellen, dass die Antlr Generator Library die ich zu erst verwendet habe, auf Antlr 4.6 basiert. Daher war der generierte Code inkompatibel mit der Antlr Runtime. Schade, da sie über einen SourceGenerator funktionierte und nicht über einen BuildTask
 * d: Ich habe das arbeiten mit AST und Parsern vertieft. (Ich habe in der Vergangenheit bereits mit der Fparsec Bibliothek gearbeitet und auch einen C# SourceGenerator geschrieben, wodurch ich Kontakt mit dem C# AST hatte)
 * e: -
 * f: https://github.com/co1inco/IFM_Programieren3/tree/master/Compilerbau/4